# Hydra Ablation Experiments

## 1. Setup and Imports

## 2. Baseline Verification

## 3. Experiment 1 - Max-Only vs. Max-and-Min

## 4. Experiment 2 - Hard vs. Soft vs. Combined Counting

## 5. Experiment 3 - First-Order Differencing On vs. Off

## 6. Experiment 4 - Clipping On vs. Off

## 7. Experiment 5 - Sensitivity Analysis

## 8. Results Summary and Visualisations

## 9. Discussion and Conclusion


In [3]:

from pathlib import Path
import sys
import time
import importlib.util

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score

# Project folders
PROJECT_DIR = Path.cwd()

REPO_DIR = PROJECT_DIR / "hydra-authors-repo"
HYDRA_SOURCE = REPO_DIR / "code" / "hydra.py"

DATA_DIR = PROJECT_DIR / "data" / "ucr_tsml"
RESULTS_DIR = PROJECT_DIR / "results" / "member3"

# Create the results folder if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Check paths before continuing
print("Project directory:", PROJECT_DIR)
print("Hydra source exists:", HYDRA_SOURCE.exists())
print("Results directory:", RESULTS_DIR)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

if not HYDRA_SOURCE.exists():
    raise FileNotFoundError(
        f"Could not find {HYDRA_SOURCE}. "
        "Check that the notebook is running from your project root."
    )

# Load the original Hydra implementation without modifying its file
spec = importlib.util.spec_from_file_location(
    "hydra_official_impl", HYDRA_SOURCE
)
hydra_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = hydra_module
spec.loader.exec_module(hydra_module)

Hydra = hydra_module.Hydra
SparseScaler = hydra_module.SparseScaler

print("\nSetup successful!")
print("Original Hydra implementation loaded.")



Project directory: c:\Users\Mahak\Desktop\Archive time sweries
Hydra source exists: True
Results directory: c:\Users\Mahak\Desktop\Archive time sweries\results\member3
PyTorch version: 2.14.1+cpu
NumPy version: 2.3.5
Pandas version: 3.0.6

Setup successful!
Original Hydra implementation loaded.


In [4]:

# Experiment 1: Max-only vs. Max-and-min

# Start with these three manageable datasets.
# We can expand the experiment after the first run works.
ABLATION_DATASETS = ["Coffee", "ECG200", "GunPoint"]

RANDOM_SEEDS = [0]
BATCH_SIZE = 256

print("Experiment 1 datasets:", ABLATION_DATASETS)
print("Random seeds:", RANDOM_SEEDS)
print("Batch size:", BATCH_SIZE)

print("\nConfiguration ready.")



Experiment 1 datasets: ['Coffee', 'ECG200', 'GunPoint']
Random seeds: [0]
Batch size: 256

Configuration ready.


In [5]:
from aeon.datasets import load_classification
import numpy as np


def load_ucr_dataset(dataset_name, extract_path=DATA_DIR):
    """
    Load the predefined train/test split for a UCR classification dataset.

    Returns
    -------
    X_train : np.ndarray
        Shape: (n_train, n_channels, n_timepoints)
    y_train : np.ndarray
        Training labels.
    X_test : np.ndarray
        Shape: (n_test, n_channels, n_timepoints)
    y_test : np.ndarray
        Test labels.
    """
    X_train, y_train = load_classification(
        name=dataset_name,
        split="train",
        extract_path=str(extract_path),
        return_metadata=False,
    )

    X_test, y_test = load_classification(
        name=dataset_name,
        split="test",
        extract_path=str(extract_path),
        return_metadata=False,
    )

    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)

    # Consistent label representation across train and test.
    y_train = np.asarray(y_train).astype(str)
    y_test = np.asarray(y_test).astype(str)

    if X_train.ndim != 3 or X_test.ndim != 3:
        raise ValueError(
            f"{dataset_name}: Expected 3D input arrays. "
            f"Got train={X_train.shape}, test={X_test.shape}"
        )

    if X_train.shape[1] != 1 or X_test.shape[1] != 1:
        raise ValueError(
            f"{dataset_name}: Expected univariate data for this "
            f"baseline. Train={X_train.shape}, test={X_test.shape}"
        )

    if X_train.shape[2] != X_test.shape[2]:
        raise ValueError(
            f"{dataset_name}: Train/test series lengths differ."
        )

    if len(X_train) != len(y_train) or len(X_test) != len(y_test):
        raise ValueError(
            f"{dataset_name}: Number of samples does not match labels."
        )

    if not set(np.unique(y_test)).issubset(set(np.unique(y_train))):
        raise ValueError(
            f"{dataset_name}: Test labels include classes absent from training."
        )

    return X_train, y_train, X_test, y_test

In [6]:

# Test that the dataset loader works correctly
X_train, y_train, X_test, y_test = load_ucr_dataset("Coffee")

print("Dataset: Coffee")
print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)
print("Training classes:", np.unique(y_train))
print("Test classes:", np.unique(y_test))



Dataset: Coffee
Training data shape: (28, 1, 286)
Training labels shape: (28,)
Test data shape: (28, 1, 286)
Test labels shape: (28,)
Training classes: ['0' '1']
Test classes: ['0' '1']


In [7]:

# Experiment 1: Inspect the original Hydra configuration

import inspect

print("Hydra constructor:")
print(inspect.signature(Hydra))

print("\nOriginal Hydra forward method:")
print(inspect.getsource(Hydra.forward))



Hydra constructor:
(input_length, k=8, g=64, seed=None)

Original Hydra forward method:
    def forward(self, X):

        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):

            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):

                _Z = F.conv1d(X if diff_index == 0 else diff_X, self.W[dilation_index, diff_index], dilation = d, padding = p) \
                      .view(num_examples, self.h, self.k, -1)

                max_values, max_indices = _Z.max(2)
                count_max = torch.zeros(num_examples, self.h, self.k)

                min_values, min_indices = _Z.min(2)
                count_min = torch.zeros(num_examples, self.h, self.k)

                count_max.scatter_add_(-1, max_indices, max_values)
                count_min.scatter_add_(-1, mi

In [8]:

class HydraMaxOnly(Hydra):

    def forward(self, X):
        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):
            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):
                input_X = X if diff_index == 0 else diff_X

                conv = torch.nn.functional.conv1d(
                    input_X,
                    self.W[dilation_index, diff_index],
                    dilation=d,
                    padding=p,
                )

                conv = conv.view(
                    num_examples, self.h, self.k, -1
                )

                max_values, max_indices = conv.max(2)

                count_max = torch.zeros(
                    num_examples,
                    self.h,
                    self.k,
                    device=conv.device,
                    dtype=conv.dtype,
                )

                count_max.scatter_add_(
                    -1, max_indices, max_values
                )

                Z.append(count_max)

        return torch.cat(Z, 1).reshape(num_examples, -1)


print("Max-only Hydra class created successfully.")



Max-only Hydra class created successfully.


In [10]:

# Load the Coffee dataset
X_train, y_train, X_test, y_test = load_ucr_dataset("Coffee")

# Use four examples for a quick test
X_sample = torch.tensor(X_train[:4], dtype=torch.float32)

# Create both models with the same settings and seed
hydra_original = Hydra(
    input_length=X_train.shape[-1],
    k=8,
    g=64,
    seed=0,
)

hydra_max_only = HydraMaxOnly(
    input_length=X_train.shape[-1],
    k=8,
    g=64,
    seed=0,
)

# Generate features without calculating gradients
with torch.no_grad():
    Z_original = hydra_original(X_sample)
    Z_max_only = hydra_max_only(X_sample)

print("Original feature shape:", tuple(Z_original.shape))
print("Max-only feature shape:", tuple(Z_max_only.shape))

assert Z_original.shape[0] == Z_max_only.shape[0]
assert Z_original.shape[1] == 2 * Z_max_only.shape[1]

print("\nSUCCESS: Both Hydra versions produced the expected feature shapes.")




Original feature shape: (4, 6144)
Max-only feature shape: (4, 3072)

SUCCESS: Both Hydra versions produced the expected feature shapes.


In [11]:

# Experiment 1: Original Hydra vs. Max-Only Hydra

def evaluate_hydra_variant(
    dataset_name,
    variant_name,
    hydra_class,
    seed=0,
    k=8,
    g=64,
    batch_size=256,
):
    # Load the same predefined train/test split
    X_train, y_train, X_test, y_test = load_ucr_dataset(dataset_name)

    start_time = time.perf_counter()

    # Convert the data to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    # Create the selected Hydra transform
    model = hydra_class(
        input_length=X_train.shape[-1],
        k=k,
        g=g,
        seed=seed,
    )

    # Transform the data in batches
    with torch.no_grad():
        Z_train = model.batch(X_train_tensor, batch_size=batch_size)
        Z_test = model.batch(X_test_tensor, batch_size=batch_size)

    # Fit a separate scaler for this variant
    scaler = SparseScaler()
    Z_train = scaler.fit_transform(Z_train)
    Z_test = scaler.transform(Z_test)

    # Train and evaluate the classifier
    classifier = RidgeClassifierCV()
    classifier.fit(Z_train.numpy(), y_train)

    predictions = classifier.predict(Z_test.numpy())

    elapsed = time.perf_counter() - start_time

    return {
        "dataset": dataset_name,
        "variant": variant_name,
        "seed": seed,
        "k": k,
        "g": g,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "runtime_seconds": elapsed,
    }


# Run the two comparisons on Coffee
results_exp1 = []

for variant_name, hydra_class in [
    ("original_max_and_min", Hydra),
    ("max_only", HydraMaxOnly),
]:
    result = evaluate_hydra_variant(
        dataset_name="Coffee",
        variant_name=variant_name,
        hydra_class=hydra_class,
        seed=0,
    )
    results_exp1.append(result)

    print(
        f"{variant_name}: "
        f"accuracy={result['accuracy']:.4f}, "
        f"macro_f1={result['macro_f1']:.4f}, "
        f"runtime={result['runtime_seconds']:.2f}s"
    )

# Save to a new file without overwriting existing results
output_path = RESULTS_DIR / "ablation_max_min.csv"
results_df = pd.DataFrame(results_exp1)

if output_path.exists():
    raise FileExistsError(
        f"{output_path} already exists. "
        "Choose a different filename to protect existing results."
    )

results_df.to_csv(output_path, index=False)

print("\nResults saved to:", output_path)
display(results_df)



original_max_and_min: accuracy=1.0000, macro_f1=1.0000, runtime=0.20s
max_only: accuracy=1.0000, macro_f1=1.0000, runtime=0.08s

Results saved to: c:\Users\Mahak\Desktop\Archive time sweries\results\member3\ablation_max_min.csv


,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,original_max_and_min,0,8,64,1.0,1.0,0.197386
1,Coffee,max_only,0,8,64,1.0,1.0,0.084144


### Experiment 1- Max-Only vs. Max-and-Min

**Dataset:** Coffee

**Original Hydra accuracy:** 100%

**Max-only Hydra accuracy:** 100%

**Original Hydra macro F1:** 1.00

**Max-only Hydra macro F1:** 1.00

**Original Hydra runtime:** 0.197 seconds

**Max-only Hydra runtime:** 0.084 seconds

**Observation:** Both variants achieved perfect classification performance on the Coffee test set. The max-only variant had a shorter measured runtime and used half as many output features as the original variant.

**Conclusion:** For this dataset and experimental run, including minimum-response features did not improve classification accuracy. Further experiments on additional datasets and seeds are required before drawing a general conclusion.

**Results file:** `results/member3/ablation_max_min.csv`


## Experiment 2 - Hard vs. Soft vs. Combined Counting

**Objective:** Investigate how the counting method used to construct Hydra features affects classification performance.

**Hard counting:** Use the original Hydra counting method as the reference.

**Soft counting:** Use a defined weighting method to distribute contributions rather than assigning them entirely to selected positions.

**Combined counting:** Combine the hard- and soft-counting feature representations.

**Evaluation:** Compare accuracy, macro F1 score, and runtime using consistent datasets, train/test splits, and classifier settings.


In [38]:
class HydraCountingVariant(Hydra):
 "Hydra with configurable hard/soft maximum and minimum counting."


def __init__(
    self,
    input_length,
    k=8,
    g=64,
    seed=None,
    max_counting="soft",
    min_counting="hard",
):
    super().__init__(input_length=input_length, k=k, g=g, seed=seed)

    if max_counting not in ("hard", "soft"):
        raise ValueError("max_counting must be 'hard' or 'soft'")

    if min_counting not in ("hard", "soft"):
        raise ValueError("min_counting must be 'hard' or 'soft'")

    self.max_counting = max_counting
    self.min_counting = min_counting

def forward(self, X):
    num_examples = X.shape[0]

    if self.divisor > 1:
        diff_X = torch.diff(X)

    Z = []

    for dilation_index in range(self.num_dilations):
        d = self.dilations[dilation_index].item()
        p = self.paddings[dilation_index].item()

        for diff_index in range(self.divisor):
            input_X = X if diff_index == 0 else diff_X

            conv = torch.nn.functional.conv1d(
                input_X,
                self.W[dilation_index, diff_index],
                dilation=d,
                padding=p,
            )

            conv = conv.view(num_examples, self.h, self.k, -1)

            # Maximum response
            max_values, max_indices = conv.max(2)
            max_counts = torch.zeros(
                num_examples, self.h, self.k,
                device=conv.device,
                dtype=conv.dtype,
            )

            if self.max_counting == "soft":
                max_updates = max_values
            else:
                max_updates = torch.ones_like(max_values)

            max_counts.scatter_add_(-1, max_indices, max_updates)
            Z.append(max_counts)

            # Minimum response
            min_values, min_indices = conv.min(2)
            min_counts = torch.zeros(
                num_examples, self.h, self.k,
                device=conv.device,
                dtype=conv.dtype,
            )

            if self.min_counting == "soft":
                min_updates = min_values
            else:
                min_updates = torch.ones_like(min_values)

            min_counts.scatter_add_(-1, min_indices, min_updates)
            Z.append(min_counts)

    return torch.cat(Z, dim=1).reshape(num_examples, -1)



In [51]:

# Test the three counting configurations
counting_variants = {
    "hard_only": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="hard",
        min_counting="hard",
    ),
    "soft_only": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="soft",
        min_counting="soft",
    ),
    "combined_default": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="soft",
        min_counting="hard",
    ),
}

dummy_X = torch.randn(4, 1, 100)

for name, model in counting_variants.items():
    with torch.no_grad():
        features = model.batch(dummy_X)

    print(f"{name}: feature shape = {tuple(features.shape)}")

print("\nAll three counting configurations ran successfully.")


print(f"{name}: feature shape = {tuple(features.shape)}")


print("\nAll three counting configurations ran successfully.")


hard_only: feature shape = (4, 4096)
soft_only: feature shape = (4, 4096)
combined_default: feature shape = (4, 4096)

All three counting configurations ran successfully.
combined_default: feature shape = (4, 4096)

All three counting configurations ran successfully.


In [69]:

# Experiment 2: Hard vs. Soft vs. Combined Counting

COUNTING_RESULTS_FILE = RESULTS_DIR / "ablation_counting_methods_v2.csv"

counting_configs = {
    "hard_only": {
        "max_counting": "hard",
        "min_counting": "hard",
    },
    "soft_only": {
        "max_counting": "soft",
        "min_counting": "soft",
    },
    "combined_default": {
        "max_counting": "soft",
        "min_counting": "hard",
    },
}

counting_results = []

for dataset_name in ABLATION_DATASETS:
    for seed in RANDOM_SEEDS:
        for variant_name, config in counting_configs.items():
            print(f"Running {dataset_name} | {variant_name} | seed={seed}")

            def make_model(**kwargs):
                return HydraCountingVariant(
                    **kwargs,
                    max_counting=config["max_counting"],
                    min_counting=config["min_counting"],
                )

            result = evaluate_hydra_variant(
                dataset_name=dataset_name,
                variant_name=variant_name,
                hydra_class=make_model,
                seed=seed,
                k=8,
                g=64,
                batch_size=BATCH_SIZE,
            )

            counting_results.append(result)

print(f"\nTotal results collected: {len(counting_results)}")

counting_df = pd.DataFrame(counting_results)

if COUNTING_RESULTS_FILE.exists():
    raise FileExistsError(
        f"Results file already exists: {COUNTING_RESULTS_FILE}. "
        "No existing results were overwritten."
    )

counting_df.to_csv(COUNTING_RESULTS_FILE, index=False)

print(f"\nResults saved to: {COUNTING_RESULTS_FILE}")
display(counting_df)



Running Coffee | hard_only | seed=0
Running Coffee | soft_only | seed=0
Running Coffee | combined_default | seed=0
Running ECG200 | hard_only | seed=0
Running ECG200 | soft_only | seed=0
Running ECG200 | combined_default | seed=0
Running GunPoint | hard_only | seed=0
Running GunPoint | soft_only | seed=0
Running GunPoint | combined_default | seed=0

Total results collected: 9

Results saved to: c:\Users\Mahak\Desktop\Archive time sweries\results\member3\ablation_counting_methods_v2.csv


,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,hard_only,0,8,64,1.00,1.000000,0.107075
1,Coffee,soft_only,0,8,64,1.00,1.000000,0.082025
2,Coffee,combined_default,0,8,64,1.00,1.000000,0.094460
3,ECG200,hard_only,0,8,64,0.84,0.816176,0.083416
4,ECG200,soft_only,0,8,64,0.90,0.890110,0.083568
5,ECG200,combined_default,0,8,64,0.89,0.878305,0.081764
6,GunPoint,hard_only,0,8,64,1.00,1.000000,0.160806
7,GunPoint,soft_only,0,8,64,1.00,1.000000,0.162458
8,GunPoint,combined_default,0,8,64,1.00,1.000000,0.172237


### Results

Three Hydra counting variants were evaluated on Coffee, ECG200, and GunPoint using seed 0, with k = 8 and g = 64.

The hard-only, soft-only, and combined methods all achieved 100% accuracy on Coffee and GunPoint. On ECG200, soft-only achieved the highest accuracy (90%), followed by combined counting (89%) and hard-only counting (84%).

### Conclusion

The results suggest that the relative performance of the counting methods may depend on the dataset. Soft-only counting performed best on ECG200, while all three methods performed equally on Coffee and GunPoint. Since the experiment used only one random seed and three datasets, additional runs are needed before drawing general conclusions.


## Experiment 3  Sensitivity to the Number of Kernels and Groups

### Objective

Investigate how different combinations of the Hydra parameters `k` and `g` affect classification accuracy, macro F1-score, and runtime.

### Experimental Setup

Three configurations will be compared while keeping `k × g = 512` constant:

| Configuration |  k |   g |
| ------------- | -: | --: |
| k4_g128       |  4 | 128 |
| k8_g64        |  8 |  64 |
| k16_g32       | 16 |  32 |

The experiment will use the Coffee, ItalyPowerDemand, and Wafer datasets, with multiple random seeds where computationally practical.

### Evaluation Metrics

* Classification accuracy
* Macro F1-score
* Runtime in seconds


In [70]:

# Experiment 3: Sensitivity to k and g

KG_RESULTS_FILE = RESULTS_DIR / "ablation_kg_sensitivity.csv"

KG_CONFIGS = [
    {"name": "k4_g128", "k": 4, "g": 128},
    {"name": "k8_g64", "k": 8, "g": 64},
    {"name": "k16_g32", "k": 16, "g": 32},
]

KG_DATASETS = ["Coffee", "ItalyPowerDemand", "Wafer"]

# Use 3 seeds for an initial comparison.
KG_SEEDS = [0, 1, 2]

kg_results = []

for dataset_name in KG_DATASETS:
    for config in KG_CONFIGS:
        for seed in KG_SEEDS:
            print(
                f"Running {dataset_name} | {config['name']} | seed={seed}"
            )

            result = evaluate_hydra_variant(
                dataset_name=dataset_name,
                variant_name=config["name"],
                hydra_class=Hydra,
                seed=seed,
                k=config["k"],
                g=config["g"],
                batch_size=BATCH_SIZE,
            )

            kg_results.append(result)

kg_df = pd.DataFrame(kg_results)

print(f"\nTotal results collected: {len(kg_df)}")
display(kg_df)

if KG_RESULTS_FILE.exists():
    raise FileExistsError(
        f"Results file already exists: {KG_RESULTS_FILE}. "
        "No existing results were overwritten."
    )

kg_df.to_csv(KG_RESULTS_FILE, index=False)
print(f"\nResults saved to: {KG_RESULTS_FILE}")



Running Coffee | k4_g128 | seed=0
Running Coffee | k4_g128 | seed=1
Running Coffee | k4_g128 | seed=2
Running Coffee | k8_g64 | seed=0
Running Coffee | k8_g64 | seed=1
Running Coffee | k8_g64 | seed=2
Running Coffee | k16_g32 | seed=0
Running Coffee | k16_g32 | seed=1
Running Coffee | k16_g32 | seed=2
Running ItalyPowerDemand | k4_g128 | seed=0
Running ItalyPowerDemand | k4_g128 | seed=1
Running ItalyPowerDemand | k4_g128 | seed=2
Running ItalyPowerDemand | k8_g64 | seed=0
Running ItalyPowerDemand | k8_g64 | seed=1
Running ItalyPowerDemand | k8_g64 | seed=2
Running ItalyPowerDemand | k16_g32 | seed=0
Running ItalyPowerDemand | k16_g32 | seed=1
Running ItalyPowerDemand | k16_g32 | seed=2
Running Wafer | k4_g128 | seed=0
Running Wafer | k4_g128 | seed=1
Running Wafer | k4_g128 | seed=2
Running Wafer | k8_g64 | seed=0
Running Wafer | k8_g64 | seed=1
Running Wafer | k8_g64 | seed=2
Running Wafer | k16_g32 | seed=0
Running Wafer | k16_g32 | seed=1
Running Wafer | k16_g32 | seed=2

Total res

,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,k4_g128,0,4,128,1.000000,1.000000,0.130540
1,Coffee,k4_g128,1,4,128,1.000000,1.000000,0.119490
2,Coffee,k4_g128,2,4,128,1.000000,1.000000,0.113416
3,Coffee,k8_g64,0,8,64,1.000000,1.000000,0.086755
4,Coffee,k8_g64,1,8,64,1.000000,1.000000,0.093425
5,Coffee,k8_g64,2,8,64,1.000000,1.000000,0.086412
6,Coffee,k16_g32,0,16,32,1.000000,1.000000,0.082813
7,Coffee,k16_g32,1,16,32,1.000000,1.000000,0.072879
8,Coffee,k16_g32,2,16,32,1.000000,1.000000,0.073461
9,ItalyPowerDemand,k4_g128,0,4,128,0.969874,0.969873,0.078532



Results saved to: c:\Users\Mahak\Desktop\Archive time sweries\results\member3\ablation_kg_sensitivity.csv


### Results

The Hydra parameter sensitivity experiment evaluated three configurations—`k4_g128`, `k8_g64`, and `k16_g32`—on Coffee, ItalyPowerDemand, and Wafer using three random seeds. The product `k × g` was held constant at 512.

Coffee achieved 100% average accuracy with all three configurations. On ItalyPowerDemand, `k8_g64` achieved the highest average accuracy, approximately 96.89%. On Wafer, all configurations achieved near-perfect average accuracy, with `k16_g32` having the lowest average runtime.

### Conclusion

The results suggest that the effect of the `k` and `g` configuration depends on the dataset. The `k16_g32` configuration was faster on Coffee and Wafer, while `k8_g64` provided the best average accuracy and runtime on ItalyPowerDemand. These findings are preliminary because only three random seeds and three datasets were evaluated.


## FINAL SUMMARY

# Member 3 — Hydra Ablation Experiments

## 1. Objective

The objective was to investigate how changes to Hydra's counting mechanisms and parameter configurations affect time-series classification performance and runtime.

## 2. Experimental Setup

Experiments were conducted using the original Hydra implementation and UCR time-series classification datasets.

The evaluation metrics were:

* Classification accuracy
* Macro F1-score
* Runtime in seconds

## 3. Experiment 1 — Maximum Counting Ablation

The original max-and-min feature representation was compared with a max-only variant on Coffee using seed 0, with k = 8 and g = 64.

Both variants achieved 100% accuracy and a macro F1-score of 1.00. The max-only variant had a shorter measured runtime in this run.

These results are preliminary because only one dataset and one seed were evaluated.

## 4. Experiment 2 — Counting Method Comparison

Hard-only, soft-only, and combined counting were evaluated on Coffee, ECG200, and GunPoint using seed 0, with k = 8 and g = 64.

All variants achieved 100% accuracy on Coffee and GunPoint. On ECG200, soft-only achieved 90% accuracy, combined counting achieved 89%, and hard-only achieved 84%.

The results suggest that the relative performance of the counting methods may depend on the dataset.

## 5. Experiment 3 — Parameter Sensitivity

Three configurations were evaluated on Coffee, ItalyPowerDemand, and Wafer using seeds 0, 1, and 2:

| Configuration |  k |   g | k × g |
| ------------- | -: | --: | ----: |
| k4_g128       |  4 | 128 |   512 |
| k8_g64        |  8 |  64 |   512 |
| k16_g32       | 16 |  32 |   512 |

Coffee achieved 100% average accuracy across all configurations. On ItalyPowerDemand, k8_g64 achieved the highest average accuracy, approximately 96.89%. On Wafer, all configurations performed near perfectly, and k16_g32 had the lowest average measured runtime.

These findings are preliminary and are limited to three datasets and three seeds.

## 6. Output Files

The experiment results are stored under `results/member3/`:

* `ablation_max_min.csv`
* `ablation_counting_methods_v2.csv`
* `ablation_kg_sensitivity.csv`

## 7. Overall Conclusion

The experiments demonstrate that Hydra's counting mechanism and parameter configuration can affect classification performance and runtime. The best-performing configuration depends on the dataset and evaluation metric.

Further experiments across additional datasets and random seeds would be needed to establish more general conclusions.
